In [ ]:
!pip install openai-whisper librosa transformers torch --quiet
!apt-get install -y ffmpeg --quiet

In [ ]:
import whisper
import librosa
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline

whisper_model = whisper.load_model("base")
'''sentiment = pipeline("text-classification",
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                     top_k=1)''' # Had to replace with other model due to language issue

In [ ]:
sentiment = pipeline("text-classification",
                     model="tabularisai/multilingual-sentiment-analysis",
                     top_k=1)

In [ ]:
def get_audio_mood(path):
    y, sr = librosa.load(path, duration=60)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    energy = np.mean(librosa.feature.rms(y=y))
    brightness = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    tempo_val = float(tempo) if np.isscalar(tempo) else float(tempo.item())
    return round((min(tempo_val/200,1) + min(float(energy)/0.1,1) + min(float(brightness)/4000,1)) / 3, 2)

def get_lyric_mood(path):
    text = whisper_model.transcribe(path)["text"][:400]
    result = sentiment(text, truncation=True, max_length=512)[0][0]
    label, score = result['label'].lower(), result['score']
    if 'positive' in label: return round(0.5 + score*0.5, 2)
    if 'negative' in label: return round(0.5 - score*0.5, 2)
    return 0.5

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
results = []
for path in uploaded:
    name = path.rsplit('.',1)[0].replace('_',' ')
    am = get_audio_mood(path)
    lm = get_lyric_mood(path)
    results.append((name, am, lm, round(abs(am-lm), 2)))
    print(f"{name}: Audio={am}, Lyrics={lm}, Dissonance={round(abs(am-lm),2)}")

In [ ]:
df = pd.DataFrame(results, columns=['Song', 'Audio', 'Lyrics', 'Dissonance'])
df.to_csv('output.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

audio_scores = df['Audio'].tolist()
lyric_scores = df['Lyrics'].tolist()
names = df['Song'].tolist()

# quadrant backgrounds
ax.axhspan(0.5, 1.0, xmin=0, xmax=0.5, alpha=0.08, color='blue')    # sounds sad, feels happy
ax.axhspan(0.5, 1.0, xmin=0.5, xmax=1.0, alpha=0.08, color='green')  # sounds happy, feels happy
ax.axhspan(0.0, 0.5, xmin=0, xmax=0.5, alpha=0.08, color='gray')     # sounds sad, feels sad
ax.axhspan(0.0, 0.5, xmin=0.5, xmax=1.0, alpha=0.08, color='red')    # sounds happy, feels sad

# quadrant labels
ax.text(0.25, 0.75, 'sounds sad\nfeels happy', ha='center', va='center',
        fontsize=9, color='blue', alpha=0.6, transform=ax.transAxes)
ax.text(0.75, 0.75, 'sounds happy\nfeels happy', ha='center', va='center',
        fontsize=9, color='green', alpha=0.6, transform=ax.transAxes)
ax.text(0.25, 0.25, 'sounds sad\nfeels sad', ha='center', va='center',
        fontsize=9, color='gray', alpha=0.6, transform=ax.transAxes)
ax.text(0.75, 0.25, 'sounds happy\nfeels sad', ha='center', va='center',
        fontsize=9, color='red', alpha=0.6, transform=ax.transAxes)

# dividing lines
ax.axhline(0.5, color='black', linewidth=0.8, alpha=0.3)
ax.axvline(0.5, color='black', linewidth=0.8, alpha=0.3)

# scatter points
ax.scatter(audio_scores, lyric_scores, s=100, zorder=5,
           color='black', edgecolors='white', linewidths=0.5)

# song labels
for i, name in enumerate(names):
    ax.annotate(name, (audio_scores[i], lyric_scores[i]),
                xytext=(8, 4), textcoords='offset points', fontsize=8)

ax.set_xlabel('Music mood (Acoustics)', fontsize=11)
ax.set_ylabel('Lyric mood (Semantics)', fontsize=11)
ax.set_title('', fontsize=13, fontweight='normal')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('quadrant_plot_1.png', dpi=400)
plt.show()